In [ ]:
# ============================================================
# 3) SENTENCE SPLIT + keyword filter -> df_sentences
# ============================================================

import numpy as np
import pandas as pd
import re

CSV_PATH = "df_articles.csv"
df_sentences_source = pd.read_csv(CSV_PATH)

if "content" not in df_sentences_source.columns:
    raise RuntimeError("Colonne 'content' absente.")

# mots-clés (phrases)
KW_SENT = ["Bürokratie",
    "Berner Verwaltung",
    "Papierkrieg",
    "Verwaltung",
    "Bundesverwaltung",
    "Beamtenapparat",
    "Amtsschimmel",
    "Regulierungsdichte",
    "Behörden",
    "Bürokraten",
    "Beamte",
    "Staatsangestellte",
    "Bureaucratie",
    "Administration publique",
    "Administration fédérale",
    "Appareil administratif",
    "Appareil étatique",
    "Appareil de l’État",
    "Autorités administratives",
    "Services de l’État",
    "Services publics",
    "Fonction publique",
    "Pouvoir administratif",
    "Autorités cantonales",
    "Administration centrale",
    "Départements fédéraux",
    "Offices fédéraux",
    "Organes de l’État",
    "Technocratie",
    "Bureaucrates",
    "Fonctionnaires",
    "Employés de l'État",
    "VBS",
    "DDPS",
    "Eidgenössische Departement für Verteidigung, Bevölkerungsschutz und Sport",
    "Département fédéral de la défense, de la protection de la population et des sports",
    "EDA",
    "DFAE",
    "Eidgenössische Departement für auswärtige Angelegenheiten",
    "Département fédéral des affaires étrangères",
    "UVEK",
    "DETEC",
    "Eidgenössische Departement für Umwelt, Verkehr, Energie und Kommunikation",
    "Département fédéral de l'environnement, des transports, de l'énergie et de la communication",
    "EJPD",
    "DFJP",
    "Eidgenössische Justiz- und Polizeidepartement",
    "Département fédéral de justice et police",
    "EDI",
    "DFI",
    "Eidgenössische Departement des Innern",
    "Département fédéral de l'intérieur",
    "EFD",
    "DFF",
    "Eidgenössische Finanzdepartement",
    "Département fédéral des finances",
    "WBF",
    "DEFR",
    "Eidgenössische Departement für Wirtschaft, Bildung und Forschung",
    "Département fédéral de l'économie, de la formation et de la recherche",]

# regex "contient un des mots-clés"
def build_kw_pattern(keywords):
    patterns = []
    for k in keywords:
        if k.isupper() and len(k) <= 4:   # acronyms like EDI, EDA, EFD
            patterns.append(rf"\b{re.escape(k)}\b")
        else:
            patterns.append(re.escape(k))
    return re.compile("|".join(patterns), flags=re.IGNORECASE)

kw_pattern = build_kw_pattern(KW_SENT)


# 1) split -> explode
tmp = df_sentences_source.copy()
tmp["sentence"] = tmp["content"].fillna("").astype(str)

# split sur . ! ? (heuristique simple)
tmp["sentence"] = tmp["sentence"].str.split(r"(?<=[.!?])\s+", regex=True)
tmp = tmp.explode("sentence", ignore_index=True)
tmp["sentence"] = tmp["sentence"].astype(str).str.strip()
tmp = tmp[tmp["sentence"].ne("")]

# 2) filtrer les phrases contenant un keyword
mask = tmp["sentence"].str.contains(kw_pattern, na=False)
tmp = tmp[mask].copy()

# 3) matched keywords (unique, join)
tmp["matched_keywords"] = tmp["sentence"].str.findall(kw_pattern).apply(
    lambda lst: ", ".join(sorted(set([x.strip() for x in lst if isinstance(x, str)])))
)

# 4) sentence_id + garder l’index article source si utile
tmp.insert(0, "sentence_id", range(1, len(tmp) + 1))
tmp["article_row_index"] = tmp.get("article_row_index", np.nan)  # optionnel si tu en avais besoin avant

# 5) df_sentences = phrases + TOUTES colonnes article propagées
# 5) df_sentences = phrases + colonnes choisies
colonnes_a_garder = [
    "sentence_id",
    "id",
    "pubtime",
    "medium_name",
    "language",
    "matched_keywords",
    "sentence",
]

df_sentences = tmp[colonnes_a_garder].copy()

print("✅ df_sentences:", df_sentences.shape)
print(df_sentences.head(5))
df_sentences.to_csv("Swissdox_sentences.csv", index=False, encoding="utf-8-sig")